In [1]:
import pandas as pd
import sqlite3
import os

In [2]:
amazon_all = pd.read_csv(
    "../data/cleaned/amazon_master_2015_2025.csv"
)

print(amazon_all.shape)

(1127609, 34)


In [3]:
os.makedirs("../database", exist_ok=True)

connection = sqlite3.connect(
    "../database/amazon_database.db"
)

print("SQLite database connected successfully!")

SQLite database connected successfully!


In [4]:
amazon_all.to_sql(
    name="amazon_sales",
    con=connection,
    if_exists="replace",
    index=False,
    chunksize=10000
)

print("Amazon data imported successfully!")

Amazon data imported successfully!


In [5]:
cursor = connection.cursor()

cursor.execute("SELECT COUNT(*) FROM amazon_sales")

print("Total records in SQL:", cursor.fetchone()[0])

Total records in SQL: 1127609


In [6]:
cursor.execute("PRAGMA table_info(amazon_sales)")

columns = cursor.fetchall()

for column in columns:
    print(column[1], "-", column[2])

transaction_id - TEXT
order_date - TEXT
customer_id - TEXT
product_id - TEXT
product_name - TEXT
category - TEXT
subcategory - TEXT
brand - TEXT
original_price_inr - REAL
discount_percent - REAL
discounted_price_inr - REAL
quantity - INTEGER
subtotal_inr - REAL
delivery_charges - REAL
final_amount_inr - REAL
customer_city - TEXT
customer_state - TEXT
customer_tier - TEXT
customer_spending_tier - TEXT
customer_age_group - TEXT
payment_method - TEXT
delivery_days - TEXT
delivery_type - TEXT
is_prime_member - TEXT
is_festival_sale - INTEGER
festival_name - TEXT
customer_rating - REAL
return_status - TEXT
order_month - INTEGER
order_year - INTEGER
order_quarter - INTEGER
product_weight_kg - REAL
is_prime_eligible - TEXT
product_rating - REAL


In [7]:
cursor.execute("""
SELECT *
FROM amazon_sales
LIMIT 5
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

('TXN_2015_00000001', '2015-01-25', 'CUST_2015_00003884', 'PROD_000021', 'Samsung Galaxy S6 16GB Black', 'Electronics', 'Smartphones', 'Samsung', 123614.29, 27.91, 89112.17, 1, 89112.17, 0.0, 89112.17, 'Mumbai', 'Maharashtra', 'Metro', 'Premium', '46-55', 'COD', '6', 'Standard', 'No', 1, 'Republic Day Sale', 5.0, 'Delivered', 1, 2015, 1, 0.19, 'True', 4.7)
('TXN_2015_00000002', '2015-01-05', 'CUST_2015_00011709', 'PROD_000055', 'OnePlus OnePlus 2 16GB White', 'Electronics', 'Smartphones', 'OnePlus', 54731.86, 0.0, 54731.86, 1, 54731.86, 0.0, 54731.86, 'Allahabad', 'Uttar Pradesh', 'Rural', 'Standard', '26-35', 'COD', '4', 'Standard', 'False', 0, 'Not Applicable', 4.5, 'Delivered', 1, 2015, 1, 0.2, 'True', 4.1)
('TXN_2015_00000003', '2015-01-24', 'CUST_2015_00004782', 'PROD_000039', 'Samsung Galaxy Note 5 64GB Black', 'Electronics', 'Smartphones', 'Samsung', 97644.25, 46.93, 51818.26, 2, 103636.52, 0.0, 103636.52, 'Mumbai', 'Maharashtra', 'Metro', 'Premium', '26-35', 'COD', '4', 'Standa

In [8]:
query = """
SELECT
    transaction_id,
    order_date,
    product_name,
    brand,
    final_amount_inr
FROM amazon_sales
LIMIT 10
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,order_date,product_name,brand,final_amount_inr
0,TXN_2015_00000001,2015-01-25,Samsung Galaxy S6 16GB Black,Samsung,89112.17
1,TXN_2015_00000002,2015-01-05,OnePlus OnePlus 2 16GB White,OnePlus,54731.86
2,TXN_2015_00000003,2015-01-24,Samsung Galaxy Note 5 64GB Black,Samsung,103636.52
3,TXN_2015_00000004,2015-01-28,Motorola Moto G (3rd Gen) 16GB Black,Motorola,21947.26
4,TXN_2015_00000005,2015-01-31,OnePlus OnePlus 2 16GB White,OnePlus,109463.72
5,TXN_2015_00000006,2015-01-04,Alienware MacBook 4GB RAM Black,Alienware,131194.65
6,TXN_2015_00000007,2015-01-27,Samsung Galaxy S6 Edge 64GB Black,Samsung,86987.64
7,TXN_2015_00000008,2015-01-08,Samsung Galaxy J7 16GB White,Samsung,32169.01
8,TXN_2015_00000009,2015-01-18,Xiaomi Redmi Note 4G 32GB White,Xiaomi,80528.32
9,TXN_2015_00000010,2015-01-03,OnePlus OnePlus 2 16GB White,OnePlus,54731.86


In [9]:
query = """
SELECT *
FROM amazon_sales
WHERE brand = 'Samsung'
LIMIT 10
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2015_00000001,2015-01-25,CUST_2015_00003884,PROD_000021,Samsung Galaxy S6 16GB Black,Electronics,Smartphones,Samsung,123614.29,27.91,...,1,Republic Day Sale,5.0,Delivered,1,2015,1,0.19,True,4.7
1,TXN_2015_00000003,2015-01-24,CUST_2015_00004782,PROD_000039,Samsung Galaxy Note 5 64GB Black,Electronics,Smartphones,Samsung,97644.25,46.93,...,1,Republic Day Sale,NaN,Delivered,1,2015,1,0.17,True,3.3
2,TXN_2015_00000007,2015-01-27,CUST_2015_00008012,PROD_000031,Samsung Galaxy S6 Edge 64GB Black,Electronics,Smartphones,Samsung,86987.64,0.00,...,0,Not Applicable,3.5,Delivered,1,2015,1,0.16,False,3.7
3,TXN_2015_00000008,2015-01-08,CUST_2015_00011809,PROD_000048,Samsung Galaxy J7 16GB White,Electronics,Smartphones,Samsung,32169.01,0.00,...,0,Not Applicable,4.0,Returned,1,2015,1,0.19,False,4.6
4,TXN_2015_00000011,2015-01-25,CUST_2015_00005519,PROD_000037,Samsung Galaxy Note 5 16GB Black,Electronics,Smartphones,Samsung,88664.85,57.92,...,1,Republic Day Sale,5.0,Delivered,1,2015,1,0.22,True,3.8
5,TXN_2015_00000012,2015-01-15,CUST_2015_00005472,PROD_001693,Samsung Galaxy Tab 8GB RAM Black,Electronics,Tablets,Samsung,73967.02,12.13,...,0,Not Applicable,4.5,Delivered,1,2015,1,0.45,True,3.7
6,TXN_2015_00000020,2015-01-11,CUST_2015_00008851,PROD_001983,Samsung OLED TV,Electronics,TV & Entertainment,Samsung,38884.19,7.56,...,0,Not Applicable,NaN,Delivered,1,2015,1,25.87,False,3.9
7,TXN_2015_00000024,2015-01-10,CUST_2015_00010162,PROD_000042,Samsung Galaxy Note 5 16GB Blue,Electronics,Smartphones,Samsung,164166.84,7.08,...,0,Not Applicable,4.5,Delivered,1,2015,1,0.16,True,3.7
8,TXN_2015_00000031,2015-01-12,CUST_2015_00010929,PROD_000028,Samsung Galaxy S6 32GB Blue,Electronics,Smartphones,Samsung,86591.57,0.00,...,0,Not Applicable,5.0,Delivered,1,2015,1,0.19,TRUE,4.5
9,TXN_2015_00000032,2015-01-18,CUST_2015_00010667,PROD_000031,Samsung Galaxy S6 Edge 64GB Black,Electronics,Smartphones,Samsung,86987.64,0.00,...,0,Not Applicable,4.5,Delivered,1,2015,1,0.16,False,3.7


In [10]:
query = """
SELECT
    transaction_id,
    product_name,
    brand,
    final_amount_inr
FROM amazon_sales
WHERE brand = 'Samsung'
AND final_amount_inr > 50000
LIMIT 10
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,brand,final_amount_inr
0,TXN_2015_00000001,Samsung Galaxy S6 16GB Black,Samsung,89112.17
1,TXN_2015_00000003,Samsung Galaxy Note 5 64GB Black,Samsung,103636.52
2,TXN_2015_00000007,Samsung Galaxy S6 Edge 64GB Black,Samsung,86987.64
3,TXN_2015_00000012,Samsung Galaxy Tab 8GB RAM Black,Samsung,64994.39
4,TXN_2015_00000020,Samsung OLED TV,Samsung,71886.04
5,TXN_2015_00000024,Samsung Galaxy Note 5 16GB Blue,Samsung,152549.75
6,TXN_2015_00000031,Samsung Galaxy S6 32GB Blue,Samsung,173183.14
7,TXN_2015_00000032,Samsung Galaxy S6 Edge 64GB Black,Samsung,86987.64
8,TXN_2015_00000037,Samsung Galaxy S6 16GB Blue,Samsung,114524.50
9,TXN_2015_00000047,Samsung Galaxy S6 32GB Blue,Samsung,74832.75


In [11]:
query = """
SELECT
    transaction_id,
    payment_method,
    final_amount_inr
FROM amazon_sales
WHERE payment_method = 'UPI'
OR payment_method = 'COD'
LIMIT 10
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,payment_method,final_amount_inr
0,TXN_2015_00000001,COD,89112.17
1,TXN_2015_00000002,COD,54731.86
2,TXN_2015_00000003,COD,103636.52
3,TXN_2015_00000004,COD,21947.26
4,TXN_2015_00000005,COD,109463.72
5,TXN_2015_00000008,COD,32169.01
6,TXN_2015_00000009,COD,80528.32
7,TXN_2015_00000010,COD,54731.86
8,TXN_2015_00000011,COD,37309.89
9,TXN_2015_00000012,COD,64994.39


In [12]:
query = """
SELECT
    payment_method,
    COUNT(*) AS total_orders
FROM amazon_sales
WHERE payment_method = 'UPI'
OR payment_method = 'COD'
GROUP BY payment_method
"""

result = pd.read_sql_query(query, connection)

result

,payment_method,total_orders
0,COD,322831
1,UPI,384228


In [13]:
query = """
SELECT
    transaction_id,
    product_name,
    brand,
    final_amount_inr
FROM amazon_sales
ORDER BY final_amount_inr DESC
LIMIT 10
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,brand,final_amount_inr
0,TXN_2021_00122346,Apple iPhone 13 mini 256GB White,Apple,1262114.31
1,TXN_2021_00025625,Apple iPhone 13 Pro Max 64GB Black,Apple,1245411.00
2,TXN_2021_00137263,Apple iPhone 13 Pro Max 64GB Black,Apple,1245411.00
3,TXN_2021_00132167,Apple iPhone 13 mini 128GB White,Apple,1232712.96
4,TXN_2021_00043827,Apple iPhone 13 Pro Max 128GB Blue,Apple,1201875.00
5,TXN_2021_00125880,Apple iPhone 13 mini 128GB Black,Apple,1198126.02
6,TXN_2021_00130446,Apple iPhone 13 mini 128GB Black,Apple,1198126.02
7,TXN_2021_00070136,Apple iPhone 13 128GB Blue,Apple,1081684.68
8,TXN_2021_00039893,Apple iPhone 13 mini 128GB White,Apple,1077960.66
9,TXN_2021_00052674,Apple iPhone 13 128GB Blue,Apple,1044675.99


In [14]:
query = """
SELECT
    transaction_id,
    product_name,
    brand,
    final_amount_inr
FROM amazon_sales
ORDER BY final_amount_inr ASC
LIMIT 10
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,brand,final_amount_inr
0,TXN_2024_00022926,OnePlus Earbuds,OnePlus,344.33
1,TXN_2022_00001268,Audio-Technica Neckband Deluxe,Audio-Technica,393.64
2,TXN_2024_00091025,OnePlus Earbuds,OnePlus,398.73
3,TXN_2025_00017237,OnePlus Earbuds,OnePlus,445.53
4,TXN_2024_00078761,Audio-Technica Neckband Deluxe,Audio-Technica,456.65
5,TXN_2021_00092969,Audio-Technica Neckband Deluxe,Audio-Technica,463.76
6,TXN_2021_00013772,Audio-Technica Neckband Deluxe,Audio-Technica,464.17
7,TXN_2025_00038027,OnePlus Earbuds,OnePlus,484.98
8,TXN_2023_00050889,Audio-Technica Neckband Deluxe,Audio-Technica,488.71
9,TXN_2024_00078787,Audio-Technica Neckband Deluxe,Audio-Technica,493.07


In [15]:
query = """
SELECT
    transaction_id,
    product_name,
    brand,
    final_amount_inr
FROM amazon_sales
WHERE brand = 'Samsung'
ORDER BY final_amount_inr DESC
LIMIT 10
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,brand,final_amount_inr
0,TXN_2021_00009840,Samsung Fire Stick,Samsung,983633.85
1,TXN_2021_00125799,Samsung Fire Stick,Samsung,983633.85
2,TXN_2021_00135736,Samsung Fire Stick,Samsung,983633.85
3,TXN_2020_00118195,Samsung Android TV Premium,Samsung,979708.77
4,TXN_2021_00136481,Samsung OLED TV Deluxe,Samsung,880849.14
5,TXN_2020_00011078,Samsung OLED TV Deluxe,Samsung,802657.78
6,TXN_2021_00020369,Samsung Smart TV,Samsung,777656.28
7,TXN_2021_00073351,Samsung Smart TV,Samsung,777656.28
8,TXN_2018_00002538,Samsung Smart TV,Samsung,752639.37
9,TXN_2022_00113087,Samsung Fire Stick,Samsung,748331.88


In [16]:
query = """
SELECT
    brand,
    SUM(final_amount_inr) AS total_revenue
FROM amazon_sales
GROUP BY brand
ORDER BY total_revenue DESC
"""

result = pd.read_sql_query(query, connection)

result

,brand,total_revenue
0,Samsung,2.062142e+10
1,Apple,1.632240e+10
2,OnePlus,1.232417e+10
3,Xiaomi,5.450159e+09
4,Realme,3.015664e+09
5,Vivo,2.348077e+09
6,Oppo,2.268237e+09
7,Lenovo,2.084388e+09
8,Alienware,1.659859e+09
9,ASUS,1.458950e+09


In [17]:
query = """
SELECT
    brand,
    COUNT(*) AS total_orders,
    SUM(final_amount_inr) AS total_revenue,
    AVG(final_amount_inr) AS average_order_value
FROM amazon_sales
GROUP BY brand
ORDER BY total_revenue DESC
"""

result = pd.read_sql_query(query, connection)

result

,brand,total_orders,total_revenue,average_order_value
0,Samsung,222664,2.062142e+10,92612.257622
1,Apple,111827,1.632240e+10,145961.141873
2,OnePlus,174329,1.232417e+10,70694.907677
3,Xiaomi,156154,5.450159e+09,34902.463043
4,Realme,84696,3.015664e+09,35605.740031
5,Vivo,71176,2.348077e+09,32989.734324
6,Oppo,71205,2.268237e+09,31855.021862
7,Lenovo,22733,2.084388e+09,91689.978852
8,Alienware,14683,1.659859e+09,113046.287272
9,ASUS,13062,1.458950e+09,111694.222056


In [18]:
query = """
SELECT
    brand,
    COUNT(*) AS total_orders,
    SUM(final_amount_inr) AS total_revenue
FROM amazon_sales
GROUP BY brand
HAVING SUM(final_amount_inr) > 10000000000
ORDER BY total_revenue DESC
"""

result = pd.read_sql_query(query, connection)

result

,brand,total_orders,total_revenue
0,Samsung,222664,2.062142e+10
1,Apple,111827,1.632240e+10
2,OnePlus,174329,1.232417e+10


In [19]:
query = """
SELECT
    brand,
    COUNT(*) AS total_orders,
    SUM(final_amount_inr) AS total_revenue
FROM amazon_sales
GROUP BY brand
HAVING COUNT(*) > 50000
AND SUM(final_amount_inr) > 10000000000
ORDER BY total_revenue DESC
"""

result = pd.read_sql_query(query, connection)

result

,brand,total_orders,total_revenue
0,Samsung,222664,2.062142e+10
1,Apple,111827,1.632240e+10
2,OnePlus,174329,1.232417e+10


In [20]:
query = """
SELECT
    transaction_id,
    product_name,
    final_amount_inr,
    CASE
        WHEN final_amount_inr >= 100000 THEN 'High Value'
        WHEN final_amount_inr >= 50000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS order_category
FROM amazon_sales
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,final_amount_inr,order_category
0,TXN_2015_00000001,Samsung Galaxy S6 16GB Black,89112.17,Medium Value
1,TXN_2015_00000002,OnePlus OnePlus 2 16GB White,54731.86,Medium Value
2,TXN_2015_00000003,Samsung Galaxy Note 5 64GB Black,103636.52,High Value
3,TXN_2015_00000004,Motorola Moto G (3rd Gen) 16GB Black,21947.26,Low Value
4,TXN_2015_00000005,OnePlus OnePlus 2 16GB White,109463.72,High Value
5,TXN_2015_00000006,Alienware MacBook 4GB RAM Black,131194.65,High Value
6,TXN_2015_00000007,Samsung Galaxy S6 Edge 64GB Black,86987.64,Medium Value
7,TXN_2015_00000008,Samsung Galaxy J7 16GB White,32169.01,Low Value
8,TXN_2015_00000009,Xiaomi Redmi Note 4G 32GB White,80528.32,Medium Value
9,TXN_2015_00000010,OnePlus OnePlus 2 16GB White,54731.86,Medium Value


In [21]:
query = """
SELECT
    CASE
        WHEN final_amount_inr >= 100000 THEN 'High Value'
        WHEN final_amount_inr >= 50000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS order_category,
    COUNT(*) AS total_orders,
    SUM(final_amount_inr) AS total_revenue
FROM amazon_sales
GROUP BY order_category
ORDER BY total_revenue DESC
"""

result = pd.read_sql_query(query, connection)

result

,order_category,total_orders,total_revenue
0,High Value,233710,4.014266e+10
1,Medium Value,275498,1.980124e+10
2,Low Value,618401,1.694483e+10


In [22]:
query = """
SELECT DISTINCT brand
FROM amazon_sales
ORDER BY brand ASC
"""

result = pd.read_sql_query(query, connection)

result

,brand
0,ASUS
1,Acer
2,Alienware
3,Amazfit
4,Apple
5,Audio-Technica
6,Boat
7,Dell
8,Fire-Boltt
9,Fitbit


In [23]:
query = """
SELECT
    COUNT(*) AS total_transactions,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(DISTINCT product_id) AS unique_products,
    COUNT(DISTINCT brand) AS unique_brands
FROM amazon_sales
"""

result = pd.read_sql_query(query, connection)

result

,total_transactions,unique_customers,unique_products,unique_brands
0,1127609,354969,2004,29


In [24]:
query="""
SELECT
    COUNT(DISTINCT customer_id)  AS unique_customers
FROM amazon_sales
"""
result = pd.read_sql_query(query,connection)

result

,unique_customers
0,354969


In [25]:
query = """
SELECT
    COUNT(*) AS total_records,
    COUNT(customer_rating) AS records_with_rating,
    COUNT(*) - COUNT(customer_rating) AS missing_ratings
FROM amazon_sales
"""

result = pd.read_sql_query(query, connection)

result

,total_records,records_with_rating,missing_ratings
0,1127609,785913,341696


In [26]:
query = """
SELECT
    transaction_id,
    product_name,
    customer_rating,
    return_status
FROM amazon_sales
WHERE customer_rating IS NULL
LIMIT 10
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,customer_rating,return_status
0,TXN_2015_00000003,Samsung Galaxy Note 5 64GB Black,None,Delivered
1,TXN_2015_00000013,OnePlus OnePlus 2 16GB Black,None,Delivered
2,TXN_2015_00000016,Audio-Technica Bluetooth Speaker Deluxe,None,Cancelled
3,TXN_2015_00000018,Apple iPhone 6 Plus 16GB Black,None,Delivered
4,TXN_2015_00000020,Samsung OLED TV,None,Delivered
5,TXN_2015_00000021,OnePlus Galaxy Tab 4GB RAM Silver,None,Delivered
6,TXN_2015_00000022,OnePlus Mi Pad 4GB RAM Silver,None,Delivered
7,TXN_2015_00000023,OnePlus OnePlus 2 16GB White,None,Delivered
8,TXN_2015_00000026,Xiaomi Redmi 2 16GB Black,None,Delivered
9,TXN_2015_00000028,LG QLED TV,None,Delivered


In [27]:
query = """
SELECT
    transaction_id,
    product_name,
    customer_rating,
    return_status
FROM amazon_sales
WHERE customer_rating IS NOT NULL
LIMIT 10
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,customer_rating,return_status
0,TXN_2015_00000001,Samsung Galaxy S6 16GB Black,5.0,Delivered
1,TXN_2015_00000002,OnePlus OnePlus 2 16GB White,4.5,Delivered
2,TXN_2015_00000004,Motorola Moto G (3rd Gen) 16GB Black,3.0,Delivered
3,TXN_2015_00000005,OnePlus OnePlus 2 16GB White,4.0,Delivered
4,TXN_2015_00000006,Alienware MacBook 4GB RAM Black,4.5,Delivered
5,TXN_2015_00000007,Samsung Galaxy S6 Edge 64GB Black,3.5,Delivered
6,TXN_2015_00000008,Samsung Galaxy J7 16GB White,4.0,Returned
7,TXN_2015_00000009,Xiaomi Redmi Note 4G 32GB White,5.0,Delivered
8,TXN_2015_00000010,OnePlus OnePlus 2 16GB White,4.5,Returned
9,TXN_2015_00000011,Samsung Galaxy Note 5 16GB Black,5.0,Delivered


In [28]:
query = """
SELECT
    transaction_id,
    payment_method,
    final_amount_inr
FROM amazon_sales
WHERE payment_method IN ('UPI', 'COD', 'Credit Card')
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,payment_method,final_amount_inr
0,TXN_2015_00000001,COD,89112.17
1,TXN_2015_00000002,COD,54731.86
2,TXN_2015_00000003,COD,103636.52
3,TXN_2015_00000004,COD,21947.26
4,TXN_2015_00000005,COD,109463.72
5,TXN_2015_00000007,Credit Card,86987.64
6,TXN_2015_00000008,COD,32169.01
7,TXN_2015_00000009,COD,80528.32
8,TXN_2015_00000010,COD,54731.86
9,TXN_2015_00000011,COD,37309.89


In [29]:
query = """
SELECT
    transaction_id,
    payment_method,
    final_amount_inr
FROM amazon_sales
WHERE payment_method NOT IN ('UPI', 'COD', 'Credit Card')
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,payment_method,final_amount_inr
0,TXN_2015_00000006,Debit Card,131194.65
1,TXN_2015_00000024,Net Banking,152549.75
2,TXN_2015_00000034,Debit Card,94029.64
3,TXN_2015_00000035,Debit Card,17206.23
4,TXN_2015_00000049,Net Banking,10066.82
5,TXN_2015_00000074,Net Banking,145716.97
6,TXN_2015_00000096,Net Banking,37535.35
7,TXN_2015_00000104,Debit Card,46458.94
8,TXN_2015_00000108,Debit Card,72795.01
9,TXN_2015_00000123,Net Banking,35329.50


In [30]:
query = """
SELECT
    transaction_id,
    product_name,
    final_amount_inr
FROM amazon_sales
WHERE final_amount_inr BETWEEN 50000 AND 100000
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,final_amount_inr
0,TXN_2015_00000001,Samsung Galaxy S6 16GB Black,89112.17
1,TXN_2015_00000002,OnePlus OnePlus 2 16GB White,54731.86
2,TXN_2015_00000007,Samsung Galaxy S6 Edge 64GB Black,86987.64
3,TXN_2015_00000009,Xiaomi Redmi Note 4G 32GB White,80528.32
4,TXN_2015_00000010,OnePlus OnePlus 2 16GB White,54731.86
5,TXN_2015_00000012,Samsung Galaxy Tab 8GB RAM Black,64994.39
6,TXN_2015_00000020,Samsung OLED TV,71886.04
7,TXN_2015_00000027,LG QLED TV,86477.53
8,TXN_2015_00000028,LG QLED TV,76837.23
9,TXN_2015_00000032,Samsung Galaxy S6 Edge 64GB Black,86987.64


In [31]:
query = """
SELECT
    transaction_id,
    product_name,
    final_amount_inr
FROM amazon_sales
WHERE final_amount_inr NOT BETWEEN 50000 AND 100000
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,final_amount_inr
0,TXN_2015_00000003,Samsung Galaxy Note 5 64GB Black,103636.52
1,TXN_2015_00000004,Motorola Moto G (3rd Gen) 16GB Black,21947.26
2,TXN_2015_00000005,OnePlus OnePlus 2 16GB White,109463.72
3,TXN_2015_00000006,Alienware MacBook 4GB RAM Black,131194.65
4,TXN_2015_00000008,Samsung Galaxy J7 16GB White,32169.01
5,TXN_2015_00000011,Samsung Galaxy Note 5 16GB Black,37309.89
6,TXN_2015_00000013,OnePlus OnePlus 2 16GB Black,35177.55
7,TXN_2015_00000014,Apple iPhone 6 16GB Blue,209875.55
8,TXN_2015_00000015,Garmin Watch Deluxe,32734.52
9,TXN_2015_00000016,Audio-Technica Bluetooth Speaker Deluxe,23075.71


In [32]:
query = """
SELECT product_name, brand
FROM amazon_sales
WHERE product_name LIKE 'Samsung%'
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,product_name,brand
0,Samsung Galaxy S6 16GB Black,Samsung
1,Samsung Galaxy Note 5 64GB Black,Samsung
2,Samsung Galaxy S6 Edge 64GB Black,Samsung
3,Samsung Galaxy J7 16GB White,Samsung
4,Samsung Galaxy Note 5 16GB Black,Samsung
5,Samsung Galaxy Tab 8GB RAM Black,Samsung
6,Samsung OLED TV,Samsung
7,Samsung Galaxy Note 5 16GB Blue,Samsung
8,Samsung Galaxy S6 32GB Blue,Samsung
9,Samsung Galaxy S6 Edge 64GB Black,Samsung


In [33]:
query = """
SELECT product_name, brand
FROM amazon_sales
WHERE product_name LIKE '%iPhone%'
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,product_name,brand
0,Apple iPhone 6 16GB Blue,Apple
1,Apple iPhone 6 Plus 16GB Black,Apple
2,Apple iPhone 6 64GB Black,Apple
3,Apple iPhone 6 Plus 64GB Black,Apple
4,Apple iPhone 5s 32GB White,Apple
5,Apple iPhone 6 Plus 32GB Blue,Apple
6,Apple iPhone 6 16GB Black,Apple
7,Apple iPhone 6 16GB White,Apple
8,Apple iPhone 6 Plus 64GB White,Apple
9,Apple iPhone 6 Plus 32GB Blue,Apple


In [35]:
query = """
SELECT
    transaction_id,
    product_name,
    COALESCE(CAST(customer_rating AS TEXT), 'Not Rated') AS rating
FROM amazon_sales
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,rating
0,TXN_2015_00000001,Samsung Galaxy S6 16GB Black,5.0
1,TXN_2015_00000002,OnePlus OnePlus 2 16GB White,4.5
2,TXN_2015_00000003,Samsung Galaxy Note 5 64GB Black,Not Rated
3,TXN_2015_00000004,Motorola Moto G (3rd Gen) 16GB Black,3.0
4,TXN_2015_00000005,OnePlus OnePlus 2 16GB White,4.0
5,TXN_2015_00000006,Alienware MacBook 4GB RAM Black,4.5
6,TXN_2015_00000007,Samsung Galaxy S6 Edge 64GB Black,3.5
7,TXN_2015_00000008,Samsung Galaxy J7 16GB White,4.0
8,TXN_2015_00000009,Xiaomi Redmi Note 4G 32GB White,5.0
9,TXN_2015_00000010,OnePlus OnePlus 2 16GB White,4.5


In [36]:
query = """
SELECT order_date
FROM amazon_sales
LIMIT 10
"""

result = pd.read_sql_query(query, connection)

result

,order_date
0,2015-01-25
1,2015-01-05
2,2015-01-24
3,2015-01-28
4,2015-01-31
5,2015-01-04
6,2015-01-27
7,2015-01-08
8,2015-01-18
9,2015-01-03


In [37]:
query = """
SELECT
    strftime('%Y', order_date) AS order_year,
    COUNT(*) AS total_orders,
    SUM(final_amount_inr) AS total_revenue
FROM amazon_sales
GROUP BY order_year
ORDER BY order_year
"""

result = pd.read_sql_query(query, connection)

result

,order_year,total_orders,total_revenue
0,2015,33165,2.142163e+09
1,2016,55275,3.598316e+09
2,2017,77385,5.510026e+09
3,2018,99495,7.248545e+09
4,2019,121605,8.605901e+09
5,2020,143715,1.187319e+10
6,2021,138187,1.099021e+10
7,2022,132660,8.532312e+09
8,2023,127132,7.712999e+09
9,2024,121605,6.823413e+09


In [38]:
query = """
SELECT brand
FROM amazon_sales
WHERE subcategory = 'Smartphones'

UNION

SELECT brand
FROM amazon_sales
WHERE subcategory = 'Laptops'
"""

result = pd.read_sql_query(query, connection)

result

,brand
0,ASUS
1,Acer
2,Alienware
3,Apple
4,Dell
5,HP
6,Lenovo
7,MSI
8,Motorola
9,Nothing


In [39]:
query = """
SELECT brand
FROM amazon_sales
WHERE subcategory = 'Smartphones'

UNION ALL

SELECT brand
FROM amazon_sales
WHERE subcategory = 'Laptops'
"""

result = pd.read_sql_query(query, connection)

result

,brand
0,Samsung
1,OnePlus
2,Samsung
3,Motorola
4,OnePlus
...,...
915542,Acer
915543,Lenovo
915544,MSI
915545,Acer


In [41]:
query = """
SELECT
    transaction_id,
    product_name,
    brand,
    final_amount_inr,
    SUM(final_amount_inr) OVER (
        PARTITION BY brand
    ) AS brand_total_revenue
FROM amazon_sales
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,brand,final_amount_inr,brand_total_revenue
0,TXN_2015_00000314,ASUS Ultrabook 4GB RAM Black,ASUS,137589.70,1.458950e+09
1,TXN_2015_00000337,ASUS Gaming 4GB RAM Black,ASUS,59626.82,1.458950e+09
2,TXN_2015_00000351,ASUS Ultrabook 4GB RAM Black,ASUS,137589.70,1.458950e+09
3,TXN_2015_00000457,ASUS Gaming 4GB RAM Black,ASUS,98893.39,1.458950e+09
4,TXN_2015_00000595,ASUS Ultrabook 4GB RAM Black,ASUS,125764.79,1.458950e+09
5,TXN_2015_00000652,ASUS Gaming 4GB RAM Black,ASUS,99841.65,1.458950e+09
6,TXN_2015_00000894,ASUS Gaming 4GB RAM Black,ASUS,94476.13,1.458950e+09
7,TXN_2015_00001010,ASUS Ultrabook 4GB RAM Black,ASUS,137589.70,1.458950e+09
8,TXN_2015_00001191,ASUS Gaming 4GB RAM Black,ASUS,84779.65,1.458950e+09
9,TXN_2015_00001357,ASUS Ultrabook 4GB RAM Black,ASUS,98653.75,1.458950e+09


In [42]:
query = """
SELECT
    transaction_id,
    product_name,
    brand,
    final_amount_inr,
    SUM(final_amount_inr) OVER(
        PARTITION BY brand
    ) AS brand_total_revenue
FROM amazon_sales
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,brand,final_amount_inr,brand_total_revenue
0,TXN_2015_00000314,ASUS Ultrabook 4GB RAM Black,ASUS,137589.70,1.458950e+09
1,TXN_2015_00000337,ASUS Gaming 4GB RAM Black,ASUS,59626.82,1.458950e+09
2,TXN_2015_00000351,ASUS Ultrabook 4GB RAM Black,ASUS,137589.70,1.458950e+09
3,TXN_2015_00000457,ASUS Gaming 4GB RAM Black,ASUS,98893.39,1.458950e+09
4,TXN_2015_00000595,ASUS Ultrabook 4GB RAM Black,ASUS,125764.79,1.458950e+09
5,TXN_2015_00000652,ASUS Gaming 4GB RAM Black,ASUS,99841.65,1.458950e+09
6,TXN_2015_00000894,ASUS Gaming 4GB RAM Black,ASUS,94476.13,1.458950e+09
7,TXN_2015_00001010,ASUS Ultrabook 4GB RAM Black,ASUS,137589.70,1.458950e+09
8,TXN_2015_00001191,ASUS Gaming 4GB RAM Black,ASUS,84779.65,1.458950e+09
9,TXN_2015_00001357,ASUS Ultrabook 4GB RAM Black,ASUS,98653.75,1.458950e+09


In [43]:
query = """
SELECT
    transaction_id,
    product_name,
    brand,
    final_amount_inr,
    SUM(final_amount_inr) OVER(
        PARTITION BY brand
    ) AS brand_total_revenue
FROM amazon_sales
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,brand,final_amount_inr,brand_total_revenue
0,TXN_2015_00000314,ASUS Ultrabook 4GB RAM Black,ASUS,137589.70,1.458950e+09
1,TXN_2015_00000337,ASUS Gaming 4GB RAM Black,ASUS,59626.82,1.458950e+09
2,TXN_2015_00000351,ASUS Ultrabook 4GB RAM Black,ASUS,137589.70,1.458950e+09
3,TXN_2015_00000457,ASUS Gaming 4GB RAM Black,ASUS,98893.39,1.458950e+09
4,TXN_2015_00000595,ASUS Ultrabook 4GB RAM Black,ASUS,125764.79,1.458950e+09
5,TXN_2015_00000652,ASUS Gaming 4GB RAM Black,ASUS,99841.65,1.458950e+09
6,TXN_2015_00000894,ASUS Gaming 4GB RAM Black,ASUS,94476.13,1.458950e+09
7,TXN_2015_00001010,ASUS Ultrabook 4GB RAM Black,ASUS,137589.70,1.458950e+09
8,TXN_2015_00001191,ASUS Gaming 4GB RAM Black,ASUS,84779.65,1.458950e+09
9,TXN_2015_00001357,ASUS Ultrabook 4GB RAM Black,ASUS,98653.75,1.458950e+09


In [44]:
query = """
SELECT
    transaction_id,
    product_name,
    brand,
    final_amount_inr,
    ROW_NUMBER() OVER (
        PARTITION BY brand
        ORDER BY final_amount_inr DESC
    ) AS row_num
FROM amazon_sales
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,brand,final_amount_inr,row_num
0,TXN_2020_00039853,ASUS Inspiron 4GB RAM Silver,ASUS,834246.60,1
1,TXN_2021_00047424,ASUS Inspiron 4GB RAM Silver,ASUS,825347.94,2
2,TXN_2021_00134842,ASUS Inspiron 4GB RAM Silver,ASUS,825347.94,3
3,TXN_2021_00043915,ASUS Ultrabook 8GB RAM Black,ASUS,789738.42,4
4,TXN_2020_00035998,ASUS Pavilion 4GB RAM Black,ASUS,731434.98,5
5,TXN_2020_00037393,ASUS Pavilion 4GB RAM Black,ASUS,731434.98,6
6,TXN_2021_00008833,ASUS Pavilion 4GB RAM Black,ASUS,723633.00,7
7,TXN_2018_00057852,ASUS Inspiron 4GB RAM Silver,ASUS,701227.80,8
8,TXN_2022_00120058,ASUS Inspiron 4GB RAM Silver,ASUS,676981.83,9
9,TXN_2021_00086577,ASUS Gaming 4GB RAM Silver,ASUS,672930.57,10


In [45]:
query = """
SELECT
    transaction_id,
    product_name,
    brand,
    final_amount_inr,
    RANK() OVER (
        PARTITION BY brand
        ORDER BY final_amount_inr DESC
    ) AS product_rank
FROM amazon_sales
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,brand,final_amount_inr,product_rank
0,TXN_2020_00039853,ASUS Inspiron 4GB RAM Silver,ASUS,834246.60,1
1,TXN_2021_00047424,ASUS Inspiron 4GB RAM Silver,ASUS,825347.94,2
2,TXN_2021_00134842,ASUS Inspiron 4GB RAM Silver,ASUS,825347.94,2
3,TXN_2021_00043915,ASUS Ultrabook 8GB RAM Black,ASUS,789738.42,4
4,TXN_2020_00035998,ASUS Pavilion 4GB RAM Black,ASUS,731434.98,5
5,TXN_2020_00037393,ASUS Pavilion 4GB RAM Black,ASUS,731434.98,5
6,TXN_2021_00008833,ASUS Pavilion 4GB RAM Black,ASUS,723633.00,7
7,TXN_2018_00057852,ASUS Inspiron 4GB RAM Silver,ASUS,701227.80,8
8,TXN_2022_00120058,ASUS Inspiron 4GB RAM Silver,ASUS,676981.83,9
9,TXN_2021_00086577,ASUS Gaming 4GB RAM Silver,ASUS,672930.57,10


In [46]:
query = """
SELECT
    transaction_id,
    product_name,
    brand,
    final_amount_inr,
    DENSE_RANK() OVER (
        PARTITION BY brand
        ORDER BY final_amount_inr DESC
    ) AS dense_rank_num
FROM amazon_sales
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,brand,final_amount_inr,dense_rank_num
0,TXN_2020_00039853,ASUS Inspiron 4GB RAM Silver,ASUS,834246.60,1
1,TXN_2021_00047424,ASUS Inspiron 4GB RAM Silver,ASUS,825347.94,2
2,TXN_2021_00134842,ASUS Inspiron 4GB RAM Silver,ASUS,825347.94,2
3,TXN_2021_00043915,ASUS Ultrabook 8GB RAM Black,ASUS,789738.42,3
4,TXN_2020_00035998,ASUS Pavilion 4GB RAM Black,ASUS,731434.98,4
5,TXN_2020_00037393,ASUS Pavilion 4GB RAM Black,ASUS,731434.98,4
6,TXN_2021_00008833,ASUS Pavilion 4GB RAM Black,ASUS,723633.00,5
7,TXN_2018_00057852,ASUS Inspiron 4GB RAM Silver,ASUS,701227.80,6
8,TXN_2022_00120058,ASUS Inspiron 4GB RAM Silver,ASUS,676981.83,7
9,TXN_2021_00086577,ASUS Gaming 4GB RAM Silver,ASUS,672930.57,8


In [47]:
query = """
SELECT
    transaction_id,
    product_name,
    final_amount_inr,
    SUM(final_amount_inr) OVER() AS overall_revenue
FROM amazon_sales
LIMIT 20
"""

result = pd.read_sql_query(query, connection)

result

,transaction_id,product_name,final_amount_inr,overall_revenue
0,TXN_2015_00000001,Samsung Galaxy S6 16GB Black,89112.17,7.688873e+10
1,TXN_2015_00000002,OnePlus OnePlus 2 16GB White,54731.86,7.688873e+10
2,TXN_2015_00000003,Samsung Galaxy Note 5 64GB Black,103636.52,7.688873e+10
3,TXN_2015_00000004,Motorola Moto G (3rd Gen) 16GB Black,21947.26,7.688873e+10
4,TXN_2015_00000005,OnePlus OnePlus 2 16GB White,109463.72,7.688873e+10
5,TXN_2015_00000006,Alienware MacBook 4GB RAM Black,131194.65,7.688873e+10
6,TXN_2015_00000007,Samsung Galaxy S6 Edge 64GB Black,86987.64,7.688873e+10
7,TXN_2015_00000008,Samsung Galaxy J7 16GB White,32169.01,7.688873e+10
8,TXN_2015_00000009,Xiaomi Redmi Note 4G 32GB White,80528.32,7.688873e+10
9,TXN_2015_00000010,OnePlus OnePlus 2 16GB White,54731.86,7.688873e+10


In [48]:
query = """
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', order_date) AS order_month,
        SUM(final_amount_inr) AS monthly_revenue
    FROM amazon_sales
    GROUP BY strftime('%Y-%m', order_date)
)

SELECT
    order_month,
    monthly_revenue,
    SUM(monthly_revenue) OVER (
        ORDER BY order_month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM monthly_revenue
ORDER BY order_month
"""

result = pd.read_sql_query(query, connection)

result

,order_month,monthly_revenue,running_total
0,2015-01,1.638995e+08,1.638995e+08
1,2015-02,1.448417e+08,3.087412e+08
2,2015-03,1.468525e+08,4.555936e+08
3,2015-04,1.767019e+08,6.322956e+08
4,2015-05,1.567187e+08,7.890142e+08
...,...,...,...
127,2025-08,3.237436e+08,7.539252e+10
128,2025-09,3.092249e+08,7.570174e+10
129,2025-10,3.148621e+08,7.601661e+10
130,2025-11,3.647337e+08,7.638134e+10


In [49]:
query = """
CREATE INDEX IF NOT EXISTS idx_customer_id
ON amazon_sales(customer_id)
"""

cursor.execute(query)
connection.commit()

print("Customer ID index created successfully!")

Customer ID index created successfully!


In [50]:
cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_order_date
ON amazon_sales(order_date)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_brand
ON amazon_sales(brand)
""")

connection.commit()

print("Additional indexes created successfully!")

Additional indexes created successfully!


In [51]:
query = """
PRAGMA index_list('amazon_sales')
"""

result = pd.read_sql_query(query, connection)
result

,seq,name,unique,origin,partial
0,0,idx_brand,0,c,0
1,1,idx_order_date,0,c,0
2,2,idx_customer_id,0,c,0


In [52]:
query = """
EXPLAIN QUERY PLAN
SELECT *
FROM amazon_sales
WHERE customer_id = 'CUST_2015_00003928'
"""

result = pd.read_sql_query(query, connection)
result

,id,parent,notused,detail
0,3,0,61,SEARCH amazon_sales USING INDEX idx_customer_i...


In [53]:
query = """
EXPLAIN QUERY PLAN
SELECT *
FROM amazon_sales
WHERE strftime('%Y', order_date) = '2020'
"""

result_a = pd.read_sql_query(query, connection)
result_a

,id,parent,notused,detail
0,2,0,216,SCAN amazon_sales


In [54]:
query = """
EXPLAIN QUERY PLAN
SELECT *
FROM amazon_sales
WHERE order_date >= '2020-01-01'
AND order_date < '2021-01-01'
"""

result_b = pd.read_sql_query(query, connection)
result_b

,id,parent,notused,detail
0,3,0,163,SEARCH amazon_sales USING INDEX idx_order_date...
